SECTION 1: SETUP

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
from datetime import datetime
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print(f"LightGBM version: {lgb.__version__}")

LightGBM version: 4.6.0


SECTION 2: CONFIGURATION

In [ ]:
DATA_DIR = r"C:\MATH699P\Data"
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed_data')
MODEL_DATA_DIR = os.path.join(PROCESSED_DIR, 'model_ready_dask')

OUTPUT_DIR = os.path.join(DATA_DIR, 'model_outputs')
MODELS_DIR = os.path.join(OUTPUT_DIR, 'models')
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results')
PLOTS_DIR = os.path.join(OUTPUT_DIR, 'plots')

for d in [OUTPUT_DIR, MODELS_DIR, RESULTS_DIR, PLOTS_DIR]:
    os.makedirs(d, exist_ok=True)
    print(f"{d}")

C:\MATH699P\Data\model_outputs
C:\MATH699P\Data\model_outputs\models
C:\MATH699P\Data\model_outputs\results
C:\MATH699P\Data\model_outputs\plots


In [ ]:
with open(os.path.join(MODEL_DATA_DIR, 'preprocessing.pkl'), 'rb') as f:
    preprocessing = pickle.load(f)

FEATURE_COLS = preprocessing['feature_cols']
TARGET_COL = preprocessing['target_col']
CONFIG = preprocessing['config']

print(f"\nFeatures: {len(FEATURE_COLS)}")
print(f"Target: {TARGET_COL}")
print(f"Train end: {CONFIG['train_end']}")
print(f"Val end: {CONFIG['val_end']}")


Features: 67
Target: OZONE
Train end: 2019-12-31
Val end: 2022-12-31


In [ ]:
print("\n" + "="*80)
print("LOADING DATA")
print("="*80)

def find_data_dir(base_dir, prefix):
    """Find the data directory."""
    candidates = [d for d in os.listdir(base_dir) 
                  if d.startswith(prefix) and os.path.isdir(os.path.join(base_dir, d))]
    if prefix in candidates:
        return os.path.join(base_dir, prefix)
    timestamped = [d for d in candidates if d != prefix]
    if timestamped:
        return os.path.join(base_dir, sorted(timestamped)[-1])
    return None

train_dir = find_data_dir(MODEL_DATA_DIR, 'train')
val_dir = find_data_dir(MODEL_DATA_DIR, 'val')
test_dir = find_data_dir(MODEL_DATA_DIR, 'test')

print(f"Train: {train_dir}")
print(f"Val:   {val_dir}")
print(f"Test:  {test_dir}")


LOADING DATA
Train: C:\MATH699P\Data\processed_data\model_ready_dask\train
Val:   C:\MATH699P\Data\processed_data\model_ready_dask\val
Test:  C:\MATH699P\Data\processed_data\model_ready_dask\test


In [ ]:
print("\nLoading training data...")
df_train = pd.read_parquet(train_dir)
X_train = df_train[FEATURE_COLS].values.astype('float32')
y_train = df_train[TARGET_COL].values.astype('float32')
print(f"Train: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")

print("Loading validation data...")
df_val = pd.read_parquet(val_dir)
X_val = df_val[FEATURE_COLS].values.astype('float32')
y_val = df_val[TARGET_COL].values.astype('float32')
print(f"Val:   {X_val.shape[0]:,} samples")

print("Loading test data...")
df_test = pd.read_parquet(test_dir)
X_test = df_test[FEATURE_COLS].values.astype('float32')
y_test = df_test[TARGET_COL].values.astype('float32')

test_site_ids = df_test['SITE_ID'].values
test_dates = df_test['DATE_TIME'].values
print(f"Test:  {X_test.shape[0]:,} samples")

del df_train, df_val
gc.collect()

print(f"\nTotal samples: {len(X_train) + len(X_val) + len(X_test):,}")


Loading training data...
Train: 18,193,749 samples, 67 features
Loading validation data...
Val:   2,072,797 samples
Loading test data...
Test:  1,400,484 samples

Total samples: 21,667,030


SECTION 3: CREATING LIGHTGBM DATASETS

In [ ]:
print("\n" + "="*80)
print("CREATING LIGHTGBM DATASETS")
print("="*80)

lgb_train = lgb.Dataset(
    X_train, 
    label=y_train,
    feature_name=FEATURE_COLS,
    free_raw_data=False
)

lgb_val = lgb.Dataset(
    X_val, 
    label=y_val,
    reference=lgb_train,
    free_raw_data=False
)

print(f"Training dataset: {X_train.shape}")
print(f"Validation dataset: {X_val.shape}")


CREATING LIGHTGBM DATASETS
Training dataset: (18193749, 67)
Validation dataset: (2072797, 67)


SECTION 4: HYPERPARAMETER TUNING WITH OPTUNA

In [ ]:
TUNE_SAMPLE_FRAC = 0.2
np.random.seed(42)
tune_idx = np.random.choice(len(X_train), int(len(X_train) * TUNE_SAMPLE_FRAC), replace=False)
X_tune = X_train[tune_idx]
y_tune = y_train[tune_idx]

print(f"Tuning on {len(X_tune):,} samples ({TUNE_SAMPLE_FRAC*100:.0f}% of training data)")

def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': 42,
        
        'num_leaves': trial.suggest_int('num_leaves', 31, 255),
        'max_depth': trial.suggest_int('max_depth', 5, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 200),
    }
    
    lgb_tune = lgb.Dataset(X_tune, label=y_tune)
    lgb_val_tune = lgb.Dataset(X_val, label=y_val, reference=lgb_tune)
    
    model = lgb.train(
        params,
        lgb_tune,
        num_boost_round=1000,  
        valid_sets=[lgb_val_tune],
        valid_names=['val'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=30),
            lgb.log_evaluation(period=0)  
        ]
    )
    
    return model.best_score['val']['rmse']

print("\nStarting hyperparameter tuning...")
print("=" * 60)

sampler = TPESampler(seed=42)
study = optuna.create_study(direction='minimize', sampler=sampler)
study.optimize(objective, n_trials=30, show_progress_bar=True)

print(f"\nBest trial:")
print(f"Value (Val RMSE): {study.best_trial.value:.4f}")
print(f"\nBest parameters:")
for key, value in study.best_trial.params.items():
    print(f"{key}: {value}")



[I 2026-02-03 17:35:51,947] A new study created in memory with name: no-name-911c295d-a24b-40e6-b9e5-442ba2edb880


Tuning on 3,638,749 samples (20% of training data)

Starting hyperparameter tuning...


  0%|          | 0/30 [00:00<?, ?it/s]

Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	val's rmse: 0.161306
[I 2026-02-03 17:39:02,966] Trial 0 finished with value: 0.16130567157918077 and parameters: {'num_leaves': 115, 'max_depth': 15, 'learning_rate': 0.05395030966670229, 'feature_fraction': 0.8394633936788146, 'bagging_fraction': 0.6624074561769746, 'bagging_freq': 2, 'lambda_l1': 3.3323645788192616e-08, 'lambda_l2': 0.6245760287469893, 'min_child_samples': 128}. Best is trial 0 with value: 0.16130567157918077.
Training until validation scores don't improve for 30 rounds
Did not meet early stopping. Best iteration is:
[1000]	val's rmse: 0.238943
[I 2026-02-03 17:41:47,488] Trial 1 finished with value: 0.23894331751234307 and parameters: {'num_leaves': 190, 'max_depth': 5, 'learning_rate': 0.09330606024425668, 'feature_fraction': 0.9329770563201687, 'bagging_fraction': 0.6849356442713105, 'bagging_freq': 2, 'lambda_l1': 4.4734294104626844e-07, 'lambda_l2

SECTION 5: MODEL TRAINING

In [ ]:
best_params = {
    'objective': 'regression',
    'metric': ['rmse', 'l1'],
    'boosting_type': 'gbdt',
    'num_leaves': 225,
    'max_depth': 14,
    'learning_rate': 0.028,
    'feature_fraction': 0.64,
    'bagging_fraction': 0.96,
    'bagging_freq': 9,
    'lambda_l1': 2.48,
    'lambda_l2': 0.00001,
    'min_child_samples': 40,
    'num_threads': -1,
    'seed': 42,
    'verbose': -1
}

NUM_BOOST_ROUNDS = 5000
EARLY_STOPPING_ROUNDS = 50

print("\n" + "=" * 60)
print("TRAINING FINAL MODEL WITH BEST PARAMETERS")
print("=" * 60)

lgb_train = lgb.Dataset(X_train, label=y_train, feature_name=FEATURE_COLS)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

start_time = datetime.now()

lgb_model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=NUM_BOOST_ROUNDS,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=EARLY_STOPPING_ROUNDS),
        lgb.log_evaluation(period=100)
    ]
)

training_time = (datetime.now() - start_time).total_seconds()

print(f"\nTraining complete!")
print(f"Best iteration: {lgb_model.best_iteration}")
print(f"Best val RMSE:  {lgb_model.best_score['val']['rmse']:.4f}")
print(f"Training time:  {training_time:.1f} seconds")


TRAINING FINAL MODEL WITH BEST PARAMETERS
Training until validation scores don't improve for 50 rounds
[100]	train's rmse: 0.971002	train's l1: 0.741796	val's rmse: 0.778835	val's l1: 0.604565
[200]	train's rmse: 0.228051	train's l1: 0.117232	val's rmse: 0.188035	val's l1: 0.0942439
[300]	train's rmse: 0.176806	train's l1: 0.0864551	val's rmse: 0.156464	val's l1: 0.0695776
[400]	train's rmse: 0.156912	train's l1: 0.0779944	val's rmse: 0.145669	val's l1: 0.0635006
[500]	train's rmse: 0.143375	train's l1: 0.0730555	val's rmse: 0.137174	val's l1: 0.0601423
[600]	train's rmse: 0.133801	train's l1: 0.0691627	val's rmse: 0.131164	val's l1: 0.0573758
[700]	train's rmse: 0.126289	train's l1: 0.0660585	val's rmse: 0.126475	val's l1: 0.0551238
[800]	train's rmse: 0.120222	train's l1: 0.063263	val's rmse: 0.122747	val's l1: 0.0530596
[900]	train's rmse: 0.114959	train's l1: 0.0609218	val's rmse: 0.119831	val's l1: 0.0513739
[1000]	train's rmse: 0.110306	train's l1: 0.0587895	val's rmse: 0.11715	

SECTION 6: SAVE MODEL

In [11]:
model_path = os.path.join(MODELS_DIR, 'lightgbm_ozone_model.txt')
lgb_model.save_model(model_path)
print(f"Model saved: {model_path}")

model_pkl_path = os.path.join(MODELS_DIR, 'lightgbm_ozone_model.pkl')
with open(model_pkl_path, 'wb') as f:
    pickle.dump(lgb_model, f)
print(f"Model pickle saved: {model_pkl_path}")

Model saved: C:\MATH699P\Data\model_outputs\models\lightgbm_ozone_model.txt
Model pickle saved: C:\MATH699P\Data\model_outputs\models\lightgbm_ozone_model.pkl


SECTION 7: EVALUATION ON TEST SET

In [ ]:
print("\n" + "="*80)
print("EVALUATION ON TEST SET")
print("="*80)

def calculate_metrics(y_true, y_pred):
    """Calculate regression metrics."""
    mask = ~(np.isnan(y_true) | np.isnan(y_pred))
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    
    nonzero = y_true != 0
    if nonzero.sum() > 0:
        mape = np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
    else:
        mape = np.nan
    
    return {'rmse': rmse, 'mae': mae, 'r2': r2, 'mape': mape}

y_pred_train = lgb_model.predict(X_train)
y_pred_val = lgb_model.predict(X_val)
y_pred_test = lgb_model.predict(X_test)

train_metrics = calculate_metrics(y_train, y_pred_train)
val_metrics = calculate_metrics(y_val, y_pred_val)
test_metrics = calculate_metrics(y_test, y_pred_test)

print("\nTraining Set:")
print(f"RMSE: {train_metrics['rmse']:.4f}, MAE: {train_metrics['mae']:.4f}, R²: {train_metrics['r2']:.4f}")

print("\nValidation Set:")
print(f"RMSE: {val_metrics['rmse']:.4f}, MAE: {val_metrics['mae']:.4f}, R²: {val_metrics['r2']:.4f}")

print("\nTest Set:")
print(f"RMSE:  {test_metrics['rmse']:.4f}")
print(f"MAE:   {test_metrics['mae']:.4f}")
print(f"R²:    {test_metrics['r2']:.4f}")
print(f"MAPE:  {test_metrics['mape']:.2f}%")


EVALUATION ON TEST SET

Training Set:
RMSE: 0.0581, MAE: 0.0306, R²: 1.0000

Validation Set:
RMSE: 0.0895, MAE: 0.0274, R²: 1.0000

Test Set:
RMSE:        0.1270
MAE:         0.0311
R²:          0.9999
MAPE:        0.13%
Skill Score: 0.9967 (vs Persistence)


SECTION 8: FEATURE IMPORTANCE

In [ ]:
print("\n" + "="*80)
print("FEATURE IMPORTANCE")
print("="*80)

importance_gain = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance_gain': lgb_model.feature_importance(importance_type='gain'),
    'importance_split': lgb_model.feature_importance(importance_type='split')
})

importance_gain = importance_gain.sort_values('importance_gain', ascending=False).reset_index(drop=True)
importance_gain['rank'] = range(1, len(importance_gain) + 1)

print("\nTop 20 Features by Gain:")
print("-" * 60)
for i, row in importance_gain.head(20).iterrows():
    print(f"{row['rank']:2d}. {row['feature']:30s} {row['importance_gain']:12.2f}")

importance_path = os.path.join(RESULTS_DIR, 'feature_importance.csv')
importance_gain.to_csv(importance_path, index=False)
print(f"\nFeature importance saved: {importance_path}")


FEATURE IMPORTANCE

Top 20 Features by Gain:
------------------------------------------------------------
 1. OZONE_rolling_mean_3           44711508988.31
 2. OZONE_lag_1                    12334732815.25
 3. OZONE_rolling_max_3            10336930907.43
 4. OZONE_rolling_min_3            5506204100.01
 5. OZONE_diff_1                   1512966660.72
 6. OZONE_velocity                 738624514.19
 7. OZONE_lag_2                    218509664.55
 8. OZONE_lag_3                    105997077.25
 9. OZONE_above_site_median         94072297.62
10. OZONE_rolling_range_3           84656276.95
11. OZONE_rolling_std_3             83103946.94
12. OZONE_rolling_max_6             66244921.61
13. OZONE_rolling_min_6             55867067.09
14. OZONE_site_p90                  54023398.59
15. OZONE_acceleration              48077319.45
16. OZONE_above_site_p75            28564032.33
17. OZONE_rolling_max_12            21713350.80
18. OZONE_site_p25                  17780518.74
19. hour_cos         

SECTION 9: SUMMARY

In [19]:
print("\n" + "="*80)
print("SAVING RESULTS SUMMARY")
print("="*80)

# Save metrics to CSV
metrics_df = pd.DataFrame([{
    'model': 'LightGBM',
    'rmse': test_metrics['rmse'],
    'mae': test_metrics['mae'],
    'r2': test_metrics['r2'],
    'mape': test_metrics['mape']
}])
metrics_df.to_csv(os.path.join(RESULTS_DIR, 'model_metrics.csv'), index=False)

# Save training report
report_path = os.path.join(RESULTS_DIR, 'training_report.txt')
with open(report_path, 'w') as f:
    f.write("="*80 + "\n")
    f.write("LIGHTGBM OZONE FORECASTING - TRAINING REPORT\n")
    f.write("="*80 + "\n\n")
    
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Training time: {training_time:.1f} seconds\n\n")
    
    f.write("-"*80 + "\n")
    f.write("DATA SUMMARY\n")
    f.write("-"*80 + "\n")
    f.write(f"Training samples:   {len(X_train):>12,}\n")
    f.write(f"Validation samples: {len(X_val):>12,}\n")
    f.write(f"Test samples:       {len(X_test):>12,}\n")
    f.write(f"Features:           {len(FEATURE_COLS):>12}\n\n")
    
    f.write("-"*80 + "\n")
    f.write("MODEL PARAMETERS\n")
    f.write("-"*80 + "\n")
    for k, v in best_params.items():
        f.write(f"{k}: {v}\n")
    f.write(f"\nBest iteration: {lgb_model.best_iteration}\n\n")
    
    f.write("-"*80 + "\n")
    f.write("TEST SET RESULTS\n")
    f.write("-"*80 + "\n")
    f.write(f"RMSE:  {test_metrics['rmse']:.4f}\n")
    f.write(f"MAE:   {test_metrics['mae']:.4f}\n")
    f.write(f"R²:    {test_metrics['r2']:.4f}\n")
    f.write(f"MAPE:  {test_metrics['mape']:.2f}%\n\n")
    
    f.write("-"*80 + "\n")
    f.write("TOP 10 FEATURES\n")
    f.write("-"*80 + "\n")
    for i, row in importance_gain.head(10).iterrows():
        f.write(f"{row['rank']:2d}. {row['feature']:<35s} {row['importance_gain']:.2f}\n")

print(f"Report saved: {report_path}")


SAVING RESULTS SUMMARY
Report saved: C:\MATH699P\Data\model_outputs\results\training_report.txt
